Import libraries and load messy dataset

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("marketing_campaign_data_messy.csv")

Cleanup header names and merge the two "clicks" columns by adding them up into a single "total_clciks" column

In [ ]:
df.columns = df.columns.str.strip()
df.columns = df.columns.str.lower()

if "clicks" in df.columns:
    df["total_clicks"] = df["clicks"].sum(axis=1)
    df = df.drop(columns="clicks")
df["total_clicks"].fillna(0)
df['total_clicks'].isna().sum() # so we have no na values in total clicks

np.int64(0)

Format column contents:

- For the "active" column convert everything to boolean objects.
- For the "channel" column fix misspelling of platforms.
- For the "spend" column make sure our date only has numbers and no symbols.
- For the "start_date" and "end_date" columns format dates so that we only keep only the date in string format.

In [ ]:
boolean_map = {
    "Y": True,
    "1": True,
    "Yes" : True,
    "0" : False,
    "No" : False
}
df["active"] = df["active"].replace(boolean_map)

apps_map = {
    "E-mail" : "Email",
    "Gogle" : "Google",
    "Tik_Tok" : "TikTok",
    "Facebok" : "Facebook",
    'Insta_gram' : "Instagram"
}
df["channel"] = df['channel'].replace(apps_map)

df["spend"] = df["spend"].str.replace('$', ' ', regex=False).str.strip()

df["start_date"] = df["start_date"].str.split(' ').str[0]
df["start_date"] = df["start_date"].str.replace('/', '-', regex=False)

parts = df["start"].str.split("-", expand=True)
print(parts)
df["start_date"] = parts[2] + "-" + parts[1] + "-" + parts[0]
df["start_date"] = pd.to_datetime(df["start_date"])

df["end_date"] = df["end_date"].str.split(' ').str[0]
df["end_date"] = df["end_date"].str.replace('/', '-', regex=False)

Check if the actual contents make sense. Mark as "NaN", cells where:
- end dates happening before start dates 
- clicks being bigger than impressions 
- conversions being bigger than clicks.

In [ ]:
bad_indices = df[df["end_date"] < df["start_date"]].index
df.loc[bad_indices, "start_date"] = np.nan
df.loc[bad_indices, "end_date"] = np.nan

bad_clicks = df[df["impressions"] < df["total_clicks"]].index
df.loc[bad_clicks, "impressions"] = np.nan
df.loc[bad_clicks, "total_clicks"] = np.nan

bad_clicks = df[df["conversions"] > df["total_clicks"]].index
df.loc[bad_clicks, "conversions"] = np.nan
df.loc[bad_clicks, "total_clicks"] = np.nan

Save our newly cleaned data into a new csv file

In [ ]:
df.to_csv("marketing_campaign_data_cleaned.csv")